# Week 3 long-source RAG corrective evaluation

This notebook reads the sanitized v1.0.0 outputs. Raw prompts, retrieved contexts, model answers, and Judge traces remain private. Automated Judge scores are diagnostic until human calibration.

In [1]:
import json
from pathlib import Path
import pandas as pd

HERE = Path.cwd()
if HERE.name != 'phase_b_evaluation':
    HERE = HERE / 'phase_b_evaluation'
analysis = json.loads((HERE / 'W03_RAG_Long_Source_Summary_v1.0.0.json').read_text(encoding='utf-8'))
summary = pd.read_csv(HERE / 'W03_RAG_Long_Source_Summary_v1.0.0.csv')
items = pd.read_csv(HERE / 'W03_RAG_Long_Source_Item_Results_v1.0.0.csv')
analysis['official_rows'], analysis['question_count'], items.shape

(80, 40, (80, 28))

In [2]:
cols = [
    'scope_value', 'n', 'required_point_coverage_mean',
    'answer_relevance_mean', 'faithfulness_mean',
    'generation_latency_ms_p50'
]
summary.loc[summary['scope'].eq('condition'), cols]

,scope_value,n,required_point_coverage_mean,answer_relevance_mean,faithfulness_mean,generation_latency_ms_p50
0,base,40,0.464583,0.041380,NaN,373.7185
1,rag,40,0.987500,0.697048,0.892811,7256.4375


In [3]:
rag = items.loc[items['condition'].eq('rag')]
rag.groupby('question_type', dropna=False).agg(
    questions=('eval_id', 'count'),
    evidence_recall=('evidence_fact_recall_at_k', 'mean'),
    coverage=('required_point_coverage', 'mean'),
    faithfulness=('faithfulness', 'mean'),
).sort_values('questions', ascending=False)

,questions,evidence_recall,coverage,faithfulness
question_type,,,,
specification_with_status,2,0.75,1.0,0.833333
architecture_synthesis,2,1.00,1.0,1.000000
model_component_boundary,2,0.50,1.0,1.000000
privacy_synthesis,2,1.00,1.0,1.000000
table_multi_field,2,1.00,1.0,1.000000
epistemic_status,2,0.75,1.0,0.205357
interface_comparison,2,1.00,1.0,1.000000
screen_coverage,1,1.00,1.0,1.000000
safety_mechanism,1,1.00,1.0,1.000000


In [4]:
pd.DataFrame(analysis['matched_rag_minus_base']).T

,finite_n,mean,p50,p95,positive,zero,negative,direction
answer_relevance,40,0.655667,0.768227,0.952463,34,5,1,rag_minus_base
required_point_coverage,40,0.522917,0.5,1.0,36,4,0,rag_minus_base
generation_latency_ms,40,8030.48255,6258.813,16423.77255,40,0,0,rag_minus_base


The matched rows compare RAG and base on the same question and generator. They do not isolate retrieval alone because retrieved content, prompt length, and generated answer length change together. See `W03_RAG_Long_Source_Corrective_Report_v1.0.0.md` for the formal interpretation.